In [1]:
import pandas as pd

In [2]:
import requests

In [3]:
import json

In [4]:
import mysql.connector

In [5]:
#COMPETITIONS DATA

In [6]:
comp_url = "https://api.sportradar.com/tennis/trial/v3/en/competitions.json?api_key=oMtq59orJ4iCh1QQ99hvlOMwoKYMpGiNSQGErOiv"

In [7]:
headers = {"accept": "application/json"}

In [8]:
response = requests.get(comp_url, headers=headers)

In [9]:
data=json.loads(response.text)

In [10]:
catdata=[]

In [11]:
compdata=[]

In [12]:
for i in data["competitions"]:
    catdata.append({
        "category_id":i["category"]["id"],
        "category_name":i["category"]["name"]
    })
    compdata.append({
          "comp_id":i["id"],
          "comp_name":i["name"],
          "parent_id":i.get("parent_id",None),
          "type":i["type"],
          "gender":i["gender"],
          "category_id":i["category"]["id"]
    })

In [13]:
df=pd.DataFrame(catdata)
df=df.where(pd.notnull(df),None)

In [14]:
df1=pd.DataFrame(compdata)
df1=df1.where(pd.notnull(df1),None)

In [15]:
#COMPLEXES DATA

In [16]:
complex_url="https://api.sportradar.com/tennis/trial/v3/en/complexes.json?api_key=oMtq59orJ4iCh1QQ99hvlOMwoKYMpGiNSQGErOiv"

In [17]:
headers = {"accept": "application/json"}

In [18]:
response2 = requests.get(complex_url, headers=headers)

In [19]:
data2=json.loads(response2.text)

In [20]:
complexdata=[]

In [21]:
venuedata=[]

In [22]:
for i in data2["complexes"]:
    complexdata.append({
        "complex_id":i["id"],
        "complex_name":i["name"]
    })
    for venue in i.get("venues",[]):
        venuedata.append({
            "venue_id":venue.get("id"),
            "venue_name":venue.get("name"),
            "city_name":venue.get("city_name"),
            "country_name":venue.get("country_name"),
            "country_code":venue.get("country_code"),
            "timezone":venue.get("timezone"),
            "complex_id":i.get("id")
        })

In [23]:
df2=pd.DataFrame(complexdata)
df2=df2.where(pd.notnull(df2),None)

In [24]:
df3=pd.DataFrame(venuedata)
df3=df3.where(pd.notnull(df3),None)

In [25]:
#RANKING DATA

In [26]:
ranking_url="https://api.sportradar.com/tennis/trial/v3/en/double_competitors_rankings.json?api_key=oMtq59orJ4iCh1QQ99hvlOMwoKYMpGiNSQGErOiv"

In [27]:
headers = {"accept": "application/json"}

In [28]:
response3 = requests.get(ranking_url, headers=headers)

In [29]:
data3=json.loads(response3.text)

In [30]:
competitor_rank=[]

In [31]:
competitor=[]

In [32]:
for i in data3.get("rankings",[]):
    for x in i.get("competitor_rankings",[]):
        y=x.get("competitor",[])
        competitor_rank.append({
         "rank_id":x.get("rank_id"),
         "rank":x.get("rank"),
         "movement":x.get("movement"),
         "points":x.get("points"),
         "competitions_played":x.get("competition_played"),
         "competitor_id":y.get("id")
        })
        competitor.append({
         "competitor_id":y.get("id"),
         "name":y.get("name"),
         "country":y.get("country"),
         "country_code":y.get("country_code"),
         "abbreviation":y.get("abbreviation")
        })

In [33]:
df4=pd.DataFrame(competitor_rank)
df4=df4.where(pd.notnull(df4),None)

In [34]:
df5=pd.DataFrame(competitor)
df5=df5.where(pd.notnull(df5),None)

In [35]:
#CONNECT SQLDATA

In [36]:
conn=mysql.connector.connect(host="localhost",user="root",password="Dhinesh@0512")

In [37]:
cursor=conn.cursor()

In [ ]:
cursor.execute("CREATE DATABASE TENNIS_DATABASE")

In [41]:
cursor.execute("USE TENNIS_DATABASE")

In [ ]:
#CREATE SQL TABLE-CATEGORIES
cursor.execute("""
    CREATE TABLE CATEGORIES(
     `Category Id` VARCHAR(50),
     `Category Name` VARCHAR(100)
    );
""")

In [42]:
#INSERTING DATAS INTO CATEGORIES TABLE
q1 = """INSERT INTO CATEGORIES(`Category Id`, `Category Name`) VALUES (%s, %s)"""
values1 = df.values.tolist()
cursor.executemany(q1, values1)
conn.commit()

In [48]:
#CREATE SQL TABLE-COMPETITIONS
cursor.execute("""
    CREATE TABLE COMPETITIONS(
     `Competition Id` VARCHAR(50),
     `Competition Name` VARCHAR(100),
     `Parent Id` VARCHAR(50),
     `Type` VARCHAR(20),
     `Gender` VARCHAR(20),
     `Category Id` VARCHAR(50)
    );
""")

In [43]:
#INSERTING DATAS INTO COMPETITIONS TABLE
q2 = """INSERT INTO COMPETITIONS(`Competition Id`, `Competition Name`,`Parent Id`,`Type`,`Gender`,`Category Id`) VALUES (%s,%s,%s,%s,%s,%s)"""
values2 = df1.values.tolist()
cursor.executemany(q2, values2)
conn.commit()

In [50]:
#CREATE SQL TABLE-COMPLEXES
cursor.execute("""
    CREATE TABLE COMPLEXES(
     `Complex Id` VARCHAR(50),
     `Complex Name` VARCHAR(100)
    );
""")

In [44]:
#INSERTING DATAS INTO COMPLEXES TABLE
q3 = """INSERT INTO COMPLEXES(`Complex Id`, `Complex Name`) VALUES (%s, %s)"""
values3 = df2.values.tolist()
cursor.executemany(q3, values3)
conn.commit()

In [52]:
#CREATE SQL TABLE-VENUES
cursor.execute("""
    CREATE TABLE VENUES(
     `Venue Id` VARCHAR(50),
     `Venue Name` VARCHAR(100),
     `City Name` VARCHAR(100),
     `Country Name` VARCHAR(100),
     `Country Code` CHAR(3),
     `Timezone`VARCHAR(100),
     `Complex Id` VARCHAR(50)
    );
""")

In [45]:
#INSERTING DATAS INTO VENUES TABLE
q4 = """INSERT INTO VENUES(`Venue Id`,`Venue Name`,`City Name`,`Country Name`,`Country Code`,`Timezone`,`Complex Id`) VALUES (%s,%s,%s,%s,%s,%s,%s)"""
values4 = df3.values.tolist()
cursor.executemany(q4, values4)
conn.commit()

In [53]:
#CREATE SQL TABLE-COMPETITOR_RANKINGS
cursor.execute("""
    CREATE TABLE COMPETITOR_RANKINGS(
     `Rank Id` INT,
     `Rank` INT,
     `Movement` INT,
     `Points` INT,
     `Competitions Played` INT,
     `Competitor Id` VARCHAR(50)
    );
""")

In [46]:
#INSERTING DATAS INTO COMPETITOR_RANKINGS TABLE
q5 = """INSERT INTO COMPETITOR_RANKINGS(`Rank Id`,`Rank`,`Movement`,`Points`,`Competitions Played`,`Competitor Id`) VALUES (%s,%s,%s,%s,%s,%s)"""
values5 = df4.values.tolist()
cursor.executemany(q5, values5)
conn.commit()

In [54]:
#CREATE SQL TABLE- COMPETITORS
cursor.execute("""
    CREATE TABLE COMPETITORS(
     `Competitor Id` VARCHAR(50),
     `Name` VARCHAR(100),
     `Country` VARCHAR(100),
     `Country Code` CHAR(3),
     `Abbreviation` VARCHAR(10)
    );
""")

In [47]:
# INSERTING DATAS INTO COMPETITORS TABLE
q6 = """INSERT INTO COMPETITORS(`Competitor Id`,`Name`,`Country`,`Country Code`,`Abbreviation`) VALUES (%s,%s,%s,%s,%s)"""
values6 = df5.values.tolist()
cursor.executemany(q6, values6)
conn.commit()

In [48]:
cursor.close()
conn.close()